In [1]:
print("hello world")

hello world


In [1]:
# ============================================================
# Cell 1 — Load Dataset & Basic Audit
# ============================================================

from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# 1. Dataset path
# ------------------------------------------------------------

DATA_PATH = Path(
    r"C:\Users\danie\supportmind-ai\rawsupportminddata\IT-helpdesk-synthetic-ticketshg.csv"
)


# ------------------------------------------------------------
# 2. Load dataset
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)


# ------------------------------------------------------------
# 3. Basic dataset information
# ------------------------------------------------------------

print("=" * 70)
print("IT HELPDESK DATASET — BASIC AUDIT")
print("=" * 70)

print(f"\nDataset path: {DATA_PATH}")
print(f"File exists: {DATA_PATH.exists()}")

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nColumns:")
for i, column in enumerate(df.columns, start=1):
    print(f"  {i}. {column}")

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
missing = df.isna().sum()
print(missing)

print("\nComplete duplicate rows:")
print(df.duplicated().sum())

print("\nUnique IDs:")
print(df["id"].nunique())

print("\nFirst 5 rows:")
display(df.head())

print("\n" + "=" * 70)
print("BASIC AUDIT COMPLETED")
print("=" * 70)

IT HELPDESK DATASET — BASIC AUDIT

Dataset path: C:\Users\danie\supportmind-ai\rawsupportminddata\IT-helpdesk-synthetic-ticketshg.csv
File exists: True

Shape: 500 rows × 7 columns

Columns:
  1. id
  2. subject
  3. description
  4. priority
  5. category
  6. createdAt
  7. requesterEmail

Data types:
id                str
subject           str
description       str
priority          str
category          str
createdAt         str
requesterEmail    str
dtype: object

Missing values:
id                0
subject           0
description       0
priority          0
category          0
createdAt         0
requesterEmail    0
dtype: int64

Complete duplicate rows:
0

Unique IDs:
500

First 5 rows:


,id,subject,description,priority,category,createdAt,requesterEmail
0,1aiu3lrqi,Hey IT! Our network printer keeps disconnecting.,Hey IT! Our network printer keeps disconnectin...,Medium,Network,2024-10-28T18:36:55.004Z,jane.doe@acme.co
1,kz5mjjpox,Re: [Acme IT] Re: Ticket #98765 - Access Issue...,This is a follow-up to your previous request #...,High,Network,2024-10-28T18:36:56.156Z,user123@acme.co
2,86eza0fwq,Software Conflict Causing App Crashes,Hey team! :wave: We're experiencing some inter...,High,Software,2024-10-28T18:36:54.644Z,user@acme.co
3,jtw509e3n,j.doe@acme.co Google Calendar Setup Assistance,We need to set up a new Google Calendar for te...,Medium,Software,2024-10-28T18:36:54.524Z,j.doe@acme.co
4,tso616mbn,Software Access: Asana Project for Jordan Smith,Jordan Smith is part of the Project Management...,Medium,Software,2024-10-28T18:36:54.816Z,jordan.smith@acme.co



BASIC AUDIT COMPLETED


In [2]:
# ============================================================
# Cell 2 — Ticket Category Distribution
# ============================================================

print("=" * 70)
print("TICKET CATEGORY DISTRIBUTION")
print("=" * 70)


# ------------------------------------------------------------
# 1. Category counts
# ------------------------------------------------------------

category_counts = (
    df["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="ticket_count")
)


# ------------------------------------------------------------
# 2. Category percentages
# ------------------------------------------------------------

category_counts["percentage"] = (
    category_counts["ticket_count"]
    / len(df)
    * 100
)


# ------------------------------------------------------------
# 3. Display distribution
# ------------------------------------------------------------

display(category_counts)


# ------------------------------------------------------------
# 4. Basic target statistics
# ------------------------------------------------------------

print(f"Number of categories: {df['category'].nunique()}")
print(f"Most common category: {category_counts.iloc[0]['category']}")
print(f"Least common category: {category_counts.iloc[-1]['category']}")

print("\n" + "=" * 70)
print("CATEGORY DISTRIBUTION COMPLETED")
print("=" * 70)

TICKET CATEGORY DISTRIBUTION


,category,ticket_count,percentage
0,Software,231,46.2
1,Account,71,14.2
2,Network,68,13.6
3,Security,42,8.4
4,Communication,24,4.8
5,Hardware,22,4.4
6,RemoteWork,20,4.0
7,Training,12,2.4
8,Infrastructure,7,1.4
9,Licensing,2,0.4


Number of categories: 11
Most common category: Software
Least common category: Performance

CATEGORY DISTRIBUTION COMPLETED


In [3]:
# ============================================================
# Cell 3 — Text / Category Sanity & Leakage Check
# ============================================================

import re


print("=" * 70)
print("TEXT / CATEGORY SANITY & LEAKAGE CHECK")
print("=" * 70)


# ------------------------------------------------------------
# 1. Create combined modelling text
# ------------------------------------------------------------

df["model_text"] = (
    df["subject"].str.strip()
    + " "
    + df["description"].str.strip()
)


# ------------------------------------------------------------
# 2. Text length statistics
# ------------------------------------------------------------

df["text_word_count"] = df["model_text"].str.split().str.len()
df["text_char_count"] = df["model_text"].str.len()

print("\nCombined text length:")
print(
    df[["text_word_count", "text_char_count"]]
    .describe()
    .round(2)
)


# ------------------------------------------------------------
# 3. Duplicate combined texts
# ------------------------------------------------------------

duplicate_text_mask = df["model_text"].duplicated(keep=False)

print("\nDuplicate combined-text rows:")
print(duplicate_text_mask.sum())

if duplicate_text_mask.sum() > 0:

    duplicate_groups = (
        df.loc[duplicate_text_mask]
        .groupby("model_text")["category"]
        .nunique()
    )

    conflicting_groups = duplicate_groups[duplicate_groups > 1]

    print(f"Duplicate text groups: {len(duplicate_groups)}")
    print(
        f"Duplicate text groups with conflicting categories: "
        f"{len(conflicting_groups)}"
    )


# ------------------------------------------------------------
# 4. Check whether category names occur in the text
# ------------------------------------------------------------

categories = sorted(df["category"].unique())

category_text_presence = []

for category in categories:

    pattern = re.escape(category.lower())

    mask = (
        df["model_text"]
        .str.lower()
        .str.contains(pattern, regex=True, na=False)
    )

    category_rows = df["category"].eq(category)

    category_text_presence.append(
        {
            "category": category,
            "tickets": int(category_rows.sum()),
            "category_name_in_text": int(
                (mask & category_rows).sum()
            ),
            "percentage": round(
                (mask & category_rows).sum()
                / category_rows.sum()
                * 100,
                2,
            ),
        }
    )


category_leakage = pd.DataFrame(category_text_presence)

print("\nCategory-name presence inside own-category text:")
display(category_leakage)


# ------------------------------------------------------------
# 5. Show representative examples
# ------------------------------------------------------------

print("\nRepresentative examples by category:")

for category in categories:

    sample = (
        df[df["category"] == category]
        [["subject", "description", "category"]]
        .head(2)
    )

    print(f"\n--- {category} ---")
    display(sample)


print("\n" + "=" * 70)
print("TEXT / CATEGORY SANITY CHECK COMPLETED")
print("=" * 70)

TEXT / CATEGORY SANITY & LEAKAGE CHECK

Combined text length:
       text_word_count  text_char_count
count           500.00           500.00
mean             66.48           453.42
std              58.56           393.82
min               9.00            64.00
25%              34.00           228.00
50%              52.00           352.50
75%              74.00           519.50
max             614.00          3545.00

Duplicate combined-text rows:
0

Category-name presence inside own-category text:


,category,tickets,category_name_in_text,percentage
0,Account,71,23,32.39
1,Communication,24,3,12.50
2,Hardware,22,5,22.73
3,Infrastructure,7,4,57.14
4,Licensing,2,2,100.00
5,Network,68,49,72.06
6,Performance,1,1,100.00
7,RemoteWork,20,0,0.00
8,Security,42,25,59.52
9,Software,231,35,15.15



Representative examples by category:

--- Account ---


,subject,description,category
6,Access Request: DB Storage Folder - john.doe@a...,Notification from Acme IT\n\nAcme IT Image\n\n...,Account
7,Assistance Needed: Password Reset for Email Ac...,I am unable to access my email account as I ha...,Account



--- Communication ---


...


--- Software ---


,subject,description,category
2,Software Conflict Causing App Crashes,Hey team! :wave: We're experiencing some inter...,Software
3,j.doe@acme.co Google Calendar Setup Assistance,We need to set up a new Google Calendar for te...,Software



--- Training ---


,subject,description,category
8,Intranet Navigation Assistance,"Hello IT Team, I've been trying to access and ...",Training
16,Google Calendar Setup Instructions,"Hi team, I'm trying to set up a personal Googl...",Training



TEXT / CATEGORY SANITY CHECK COMPLETED


In [4]:
# ============================================================
# Cell 4 — Target-Specific Text Preprocessing
# ============================================================

import re


def clean_ticket_text(subject: str, description: str) -> str:
    """
    Combine subject and description while removing only
    administrative/identifying noise that is not useful
    for ticket-category classification.
    """

    text = f"{subject} {description}"

    # --------------------------------------------------------
    # 1. Remove email addresses
    # --------------------------------------------------------
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        " ",
        text,
    )

    # --------------------------------------------------------
    # 2. Remove URLs
    # --------------------------------------------------------
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 3. Remove common ticket/reference identifiers
    #    Examples:
    #    Ticket #98765
    #    Ticket ID: ABC123
    #    Request #12345
    # --------------------------------------------------------
    text = re.sub(
        r"\b(?:ticket|request|case|incident)\s*(?:id)?\s*#?\s*[A-Za-z0-9-]+\b",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 4. Normalize whitespace
    # --------------------------------------------------------
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ------------------------------------------------------------
# Apply preprocessing
# ------------------------------------------------------------

df["clean_text"] = df.apply(
    lambda row: clean_ticket_text(
        row["subject"],
        row["description"],
    ),
    axis=1,
)


# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("=" * 70)
print("TARGET-SPECIFIC TEXT PREPROCESSING")
print("=" * 70)

print(f"\nOriginal text rows: {len(df):,}")
print(f"Clean text rows:    {df['clean_text'].notna().sum():,}")

print(
    f"\nEmpty cleaned texts: "
    f"{df['clean_text'].str.strip().eq('').sum():,}"
)

print(
    f"Duplicate cleaned texts: "
    f"{df['clean_text'].duplicated().sum():,}"
)

print("\nBefore → After examples:")

display(
    df[
        ["subject", "description", "clean_text", "category"]
    ].head(10)
)

print("\n" + "=" * 70)
print("TEXT PREPROCESSING COMPLETED")
print("=" * 70)

TARGET-SPECIFIC TEXT PREPROCESSING

Original text rows: 500
Clean text rows:    500

Empty cleaned texts: 0
Duplicate cleaned texts: 0

Before → After examples:


,subject,description,clean_text,category
0,Hey IT! Our network printer keeps disconnecting.,Hey IT! Our network printer keeps disconnectin...,Hey IT! Our network printer keeps disconnectin...,Network
1,Re: [Acme IT] Re: Ticket #98765 - Access Issue...,This is a follow-up to your previous request #...,Re: [Acme IT] Re: - Access Issue with Shared N...,Network
2,Software Conflict Causing App Crashes,Hey team! :wave: We're experiencing some inter...,Software Conflict Causing App Crashes Hey team...,Software
3,j.doe@acme.co Google Calendar Setup Assistance,We need to set up a new Google Calendar for te...,Google Calendar Setup Assistance We need to se...,Software
4,Software Access: Asana Project for Jordan Smith,Jordan Smith is part of the Project Management...,Software Access: Asana Project for Jordan Smit...,Software
5,Email client sync issue,Email client fails to sync with server due to ...,Email client sync issue Email client fails to ...,Software
6,Access Request: DB Storage Folder - john.doe@a...,Notification from Acme IT\n\nAcme IT Image\n\n...,Access Request: DB Storage Folder - Notificati...,Account
7,Assistance Needed: Password Reset for Email Ac...,I am unable to access my email account as I ha...,Assistance Needed: Password Reset for Email Ac...,Account
8,Intranet Navigation Assistance,"Hello IT Team, I've been trying to access and ...","Intranet Navigation Assistance Hello IT Team, ...",Training
9,Workstation Security Configuration: Inconsiste...,"Hello IT Team, I've noticed that the group pol...",Workstation Security Configuration: Inconsiste...,Security



TEXT PREPROCESSING COMPLETED


In [6]:
# ============================================================
# Cell 5 — Add Limited Synthetic Examples for Rare Class
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# Synthetic examples for the extremely rare Performance class
# ------------------------------------------------------------

synthetic_performance = pd.DataFrame(
    [
        {
            "id": "SYN_PERF_001",
            "subject": "Application Performance Degradation",
            "description": (
                "The application has become very slow during normal use. "
                "Pages take several seconds to load and overall system "
                "performance has noticeably degraded."
            ),
            "priority": "Medium",
            "category": "Performance",
            "createdAt": df["createdAt"].iloc[0],
            "requesterEmail": "synthetic_performance_001@supportmind.local",
        },
        {
            "id": "SYN_PERF_002",
            "subject": "System Running Slowly",
            "description": (
                "The system response time is significantly slower than usual. "
                "Applications take a long time to open and actions are delayed "
                "even though there are no reported connectivity issues."
            ),
            "priority": "Medium",
            "category": "Performance",
            "createdAt": df["createdAt"].iloc[0],
            "requesterEmail": "synthetic_performance_002@supportmind.local",
        },
        {
            "id": "SYN_PERF_003",
            "subject": "Slow Application Response",
            "description": (
                "Users are experiencing poor application performance. "
                "The software responds slowly to commands and takes longer "
                "than expected to complete normal operations."
            ),
            "priority": "Medium",
            "category": "Performance",
            "createdAt": df["createdAt"].iloc[0],
            "requesterEmail": "synthetic_performance_003@supportmind.local",
        },
    ]
)


# ------------------------------------------------------------
# Append synthetic examples
# ------------------------------------------------------------

df = pd.concat(
    [df, synthetic_performance],
    ignore_index=True,
)


# ------------------------------------------------------------
# Recreate clean text for synthetic rows
# ------------------------------------------------------------

df["clean_text"] = df.apply(
    lambda row: clean_ticket_text(
        row["subject"],
        row["description"],
    ),
    axis=1,
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("LIMITED SYNTHETIC DATA ADDITION")
print("=" * 70)

print(f"\nDataset shape: {df.shape}")

print("\nPerformance class:")
print(
    df[df["category"] == "Performance"][
        ["id", "subject", "description", "category"]
    ].to_string(index=False)
)

print("\nUpdated category distribution:")
print(df["category"].value_counts())

print("\n" + "=" * 70)
print("3 SYNTHETIC PERFORMANCE EXAMPLES ADDED")
print("=" * 70)

LIMITED SYNTHETIC DATA ADDITION

Dataset shape: (503, 11)

Performance class:
          id                                     subject                                                                                                                                                                                                                                                                                                                                                                                                 description    category
   f3czsohfi Enterprise app performance monitoring issue Hi IT,\n\nI'm experiencing persistent issues with our enterprise application performance monitoring system. Specific application metrics intermittently fail to report, which requires a detailed investigation and manual adjustment of the monitoring configurations. I might also need to inspect the application server logs.\n\nCould you please advise on the next steps? Thank you.\n\nCheers,\nAlex Perf

In [8]:
# ============================================================
# Cell 6 — Train / Validation / Test Split
# ============================================================

from sklearn.model_selection import train_test_split


# ------------------------------------------------------------
# Separate original and synthetic data
# ------------------------------------------------------------

synthetic_mask = df["id"].astype(str).str.startswith("SYN_")

df_real = df[~synthetic_mask].copy()
df_synthetic_existing = df[synthetic_mask].copy()


# ------------------------------------------------------------
# Split REAL data
#
# We use a two-stage split, but only stratify classes that
# have enough examples to support the requested split.
# ------------------------------------------------------------

category_counts = df_real["category"].value_counts()

stratifiable_categories = category_counts[
    category_counts >= 10
].index.tolist()

df_stratifiable = df_real[
    df_real["category"].isin(stratifiable_categories)
].copy()

df_rare = df_real[
    ~df_real["category"].isin(stratifiable_categories)
].copy()


# ------------------------------------------------------------
# 80% train / 20% temporary
# ------------------------------------------------------------

train_main, temp_main = train_test_split(
    df_stratifiable,
    test_size=0.20,
    stratify=df_stratifiable["category"],
    random_state=42,
)


# ------------------------------------------------------------
# 10% validation / 10% test
#
# At this stage, some smaller classes may have too few
# examples for another stratified split.
#
# Therefore we split the temporary set WITHOUT stratification.
# The first split already preserved the major class distribution.
# ------------------------------------------------------------

val_main, test_main = train_test_split(
    temp_main,
    test_size=0.50,
    random_state=42,
)


# ------------------------------------------------------------
# Put rare real categories into the training set.
#
# These classes are too small to produce reliable separate
# validation/test estimates.
# ------------------------------------------------------------

train_df = pd.concat(
    [
        train_main,
        df_rare,
        df_synthetic_existing,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# Validation and test contain REAL tickets only
# ------------------------------------------------------------

val_df = val_main.reset_index(drop=True)
test_df = test_main.reset_index(drop=True)


# ------------------------------------------------------------
# Shuffle training data
# ------------------------------------------------------------

train_df = train_df.sample(
    frac=1,
    random_state=42,
).reset_index(drop=True)


# ------------------------------------------------------------
# Extract model inputs and targets
# ------------------------------------------------------------

X_train = train_df["clean_text"]
y_train = train_df["category"]

X_val = val_df["clean_text"]
y_val = val_df["category"]

X_test = test_df["clean_text"]
y_test = test_df["category"]


# ------------------------------------------------------------
# Validation output
# ------------------------------------------------------------

print("=" * 70)
print("TRAIN / VALIDATION / TEST SPLIT")
print("=" * 70)

print(f"\nReal dataset rows : {len(df_real):,}")
print(f"Existing synthetic rows : {len(df_synthetic_existing):,}")

print(f"\nTrain      : {len(train_df):,}")
print(f"Validation : {len(val_df):,}")
print(f"Test       : {len(test_df):,}")


print("\nTrain distribution:")
print(y_train.value_counts())


print("\nValidation distribution:")
print(y_val.value_counts())


print("\nTest distribution:")
print(y_test.value_counts())


# ------------------------------------------------------------
# Check which categories appear in each split
# ------------------------------------------------------------

print("\nCategory presence across splits:")

all_categories = sorted(df_real["category"].unique())

for category in all_categories:

    train_count = (y_train == category).sum()
    val_count = (y_val == category).sum()
    test_count = (y_test == category).sum()

    print(
        f"{category:<18} "
        f"Train={train_count:<3} "
        f"Val={val_count:<3} "
        f"Test={test_count:<3}"
    )


print("\n" + "=" * 70)
print("SPLIT COMPLETED")
print("=" * 70)

TRAIN / VALIDATION / TEST SPLIT

Real dataset rows : 500
Existing synthetic rows : 3

Train      : 405
Validation : 49
Test       : 49

Train distribution:
category
Software          185
Account            57
Network            54
Security           34
Communication      19
Hardware           17
RemoteWork         16
Training           10
Infrastructure      7
Performance         4
Licensing           2
Name: count, dtype: int64

Validation distribution:
category
Software         22
Account           9
Network           7
Hardware          4
Security          3
Communication     2
Training          1
RemoteWork        1
Name: count, dtype: int64

Test distribution:
category
Software         24
Network           7
Account           5
Security          5
Communication     3
RemoteWork        3
Hardware          1
Training          1
Name: count, dtype: int64

Category presence across splits:
Account            Train=57  Val=9   Test=5  
Communication      Train=19  Val=2   Test=3  
Hardw

In [9]:
# ============================================================
# Cell 7 — TF-IDF + Logistic Regression
#             Class-Weighted Baseline
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


# ------------------------------------------------------------
# TF-IDF
# ------------------------------------------------------------

tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)


# ------------------------------------------------------------
# IMPORTANT:
# Fit TF-IDF ONLY on training data
# ------------------------------------------------------------

X_train_tfidf = tfidf.fit_transform(X_train)

X_val_tfidf = tfidf.transform(X_val)

X_test_tfidf = tfidf.transform(X_test)


# ------------------------------------------------------------
# Logistic Regression with class weighting
# ------------------------------------------------------------

lr_balanced = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    C=1.0,
    random_state=42,
)


# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

lr_balanced.fit(
    X_train_tfidf,
    y_train,
)


# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

y_val_pred = lr_balanced.predict(X_val_tfidf)

y_test_pred = lr_balanced.predict(X_test_tfidf)


# ------------------------------------------------------------
# Basic output
# ------------------------------------------------------------

print("=" * 70)
print("TF-IDF + LOGISTIC REGRESSION")
print("=" * 70)

print(f"\nTraining samples   : {X_train_tfidf.shape[0]:,}")
print(f"TF-IDF features    : {X_train_tfidf.shape[1]:,}")

print("\nClass weighting:")
print("class_weight='balanced'")

print("\nModel training completed.")

print("\nValidation predictions:", len(y_val_pred))
print("Test predictions      :", len(y_test_pred))

print("\n" + "=" * 70)
print("BASELINE TRAINING COMPLETED")
print("=" * 70)

TF-IDF + LOGISTIC REGRESSION

Training samples   : 405
TF-IDF features    : 3,846

Class weighting:
class_weight='balanced'

Model training completed.

Validation predictions: 49
Test predictions      : 49

BASELINE TRAINING COMPLETED


In [10]:
# ============================================================
# Cell 8 — Validation Evaluation
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)


# ------------------------------------------------------------
# Validation metrics
# ------------------------------------------------------------

val_accuracy = accuracy_score(
    y_val,
    y_val_pred,
)

val_macro_f1 = f1_score(
    y_val,
    y_val_pred,
    average="macro",
    zero_division=0,
)

val_weighted_f1 = f1_score(
    y_val,
    y_val_pred,
    average="weighted",
    zero_division=0,
)


# ------------------------------------------------------------
# Print overall metrics
# ------------------------------------------------------------

print("=" * 70)
print("VALIDATION EVALUATION")
print("=" * 70)

print(f"\nAccuracy    : {val_accuracy:.4f}")
print(f"Macro-F1    : {val_macro_f1:.4f}")
print(f"Weighted-F1 : {val_weighted_f1:.4f}")


# ------------------------------------------------------------
# Per-class report
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("PER-CLASS CLASSIFICATION REPORT")
print("-" * 70)

print(
    classification_report(
        y_val,
        y_val_pred,
        zero_division=0,
    )
)

print("=" * 70)

VALIDATION EVALUATION

Accuracy    : 0.5918
Macro-F1    : 0.5789
Weighted-F1 : 0.5859

----------------------------------------------------------------------
PER-CLASS CLASSIFICATION REPORT
----------------------------------------------------------------------
               precision    recall  f1-score   support

      Account       0.57      0.89      0.70         9
Communication       0.25      0.50      0.33         2
     Hardware       0.75      0.75      0.75         4
      Network       0.71      0.71      0.71         7
   RemoteWork       0.50      1.00      0.67         1
     Security       0.50      0.67      0.57         3
     Software       0.80      0.36      0.50        22
     Training       0.25      1.00      0.40         1

     accuracy                           0.59        49
    macro avg       0.54      0.74      0.58        49
 weighted avg       0.68      0.59      0.59        49



In [11]:
# ============================================================
# Cell 9 — Inspect Training Class Imbalance
# ============================================================

print("=" * 70)
print("CURRENT TRAINING CLASS DISTRIBUTION")
print("=" * 70)

train_counts = y_train.value_counts()

print("\nCategory counts:")
print(train_counts)

print("\nPercentage distribution:")
print(
    (train_counts / len(y_train) * 100)
    .round(2)
    .sort_values(ascending=False)
)

print("\nMinimum training class size:", train_counts.min())
print("Maximum training class size:", train_counts.max())

print("\n" + "=" * 70)

CURRENT TRAINING CLASS DISTRIBUTION

Category counts:
category
Software          185
Account            57
Network            54
Security           34
Communication      19
Hardware           17
RemoteWork         16
Training           10
Infrastructure      7
Performance         4
Licensing           2
Name: count, dtype: int64

Percentage distribution:
category
Software          45.68
Account           14.07
Network           13.33
Security           8.40
Communication      4.69
Hardware           4.20
RemoteWork         3.95
Training           2.47
Infrastructure     1.73
Performance        0.99
Licensing          0.49
Name: count, dtype: float64

Minimum training class size: 2
Maximum training class size: 185



In [12]:
# ============================================================
# Cell 10 — Moderate Oversampling of Training Data
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# Target size for minority classes
# ------------------------------------------------------------

OVERSAMPLE_TARGET = 30


# ------------------------------------------------------------
# Current training distribution
# ------------------------------------------------------------

print("=" * 70)
print("MODERATE TRAINING OVERSAMPLING")
print("=" * 70)

print("\nOriginal training distribution:")
print(y_train.value_counts())


# ------------------------------------------------------------
# Oversample minority classes
#
# Only TRAINING data is modified.
# Validation and test remain completely untouched.
# ------------------------------------------------------------

balanced_train_parts = []

for category, group in train_df.groupby("category"):

    current_count = len(group)

    if current_count < OVERSAMPLE_TARGET:

        additional_count = OVERSAMPLE_TARGET - current_count

        extra_samples = group.sample(
            n=additional_count,
            replace=True,
            random_state=42,
        )

        group_balanced = pd.concat(
            [group, extra_samples],
            ignore_index=True,
        )

    else:
        group_balanced = group.copy()

    balanced_train_parts.append(group_balanced)


# ------------------------------------------------------------
# Combine classes
# ------------------------------------------------------------

train_df_oversampled = pd.concat(
    balanced_train_parts,
    ignore_index=True,
)


# ------------------------------------------------------------
# Shuffle
# ------------------------------------------------------------

train_df_oversampled = train_df_oversampled.sample(
    frac=1,
    random_state=42,
).reset_index(drop=True)


# ------------------------------------------------------------
# Extract oversampled training data
# ------------------------------------------------------------

X_train_over = train_df_oversampled["clean_text"]
y_train_over = train_df_oversampled["category"]


# ------------------------------------------------------------
# Validation / test remain unchanged
# ------------------------------------------------------------

X_val_over = X_val
y_val_over = y_val

X_test_over = X_test
y_test_over = y_test


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("OVERSAMPLED TRAINING DISTRIBUTION")
print("-" * 70)

print(y_train_over.value_counts())


print("\nTraining size:")
print(f"Before oversampling : {len(train_df):,}")
print(f"After oversampling  : {len(train_df_oversampled):,}")

print("\nValidation size:", len(X_val_over))
print("Test size      :", len(X_test_over))


# ------------------------------------------------------------
# Verify validation/test were not modified
# ------------------------------------------------------------

print("\nValidation/Test protection:")
print(f"Validation rows : {len(X_val_over):,}")
print(f"Test rows       : {len(X_test_over):,}")

print("\n" + "=" * 70)
print("MODERATE OVERSAMPLING COMPLETED")
print("=" * 70)

MODERATE TRAINING OVERSAMPLING

Original training distribution:
category
Software          185
Account            57
Network            54
Security           34
Communication      19
Hardware           17
RemoteWork         16
Training           10
Infrastructure      7
Performance         4
Licensing           2
Name: count, dtype: int64

----------------------------------------------------------------------
OVERSAMPLED TRAINING DISTRIBUTION
----------------------------------------------------------------------
category
Software          185
Account            57
Network            54
Security           34
Communication      30
Training           30
Infrastructure     30
RemoteWork         30
Hardware           30
Licensing          30
Performance        30
Name: count, dtype: int64

Training size:
Before oversampling : 405
After oversampling  : 540

Validation size: 49
Test size      : 49

Validation/Test protection:
Validation rows : 49
Test rows       : 49

MODERATE OVERSAMPLING CO

In [13]:
# ============================================================
# Cell 11 — TF-IDF + Logistic Regression
#             Moderate Oversampling
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


# ------------------------------------------------------------
# TF-IDF
# ------------------------------------------------------------

tfidf_over = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)


# ------------------------------------------------------------
# IMPORTANT:
# Fit TF-IDF ONLY on oversampled training data
# ------------------------------------------------------------

X_train_over_tfidf = tfidf_over.fit_transform(
    X_train_over
)

X_val_over_tfidf = tfidf_over.transform(
    X_val_over
)

X_test_over_tfidf = tfidf_over.transform(
    X_test_over
)


# ------------------------------------------------------------
# Logistic Regression
#
# No class weighting because the training data has already
# been moderately oversampled.
# ------------------------------------------------------------

lr_oversampled = LogisticRegression(
    class_weight=None,
    max_iter=2000,
    C=1.0,
    random_state=42,
)


# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

lr_oversampled.fit(
    X_train_over_tfidf,
    y_train_over,
)


# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

y_val_pred_over = lr_oversampled.predict(
    X_val_over_tfidf
)

y_test_pred_over = lr_oversampled.predict(
    X_test_over_tfidf
)


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("=" * 70)
print("TF-IDF + LOGISTIC REGRESSION — OVERSAMPLED")
print("=" * 70)

print(f"\nTraining samples : {X_train_over_tfidf.shape[0]:,}")
print(f"TF-IDF features  : {X_train_over_tfidf.shape[1]:,}")

print("\nOversampling target : 30")
print("class_weight       : None")

print("\nModel training completed.")

print(
    f"\nValidation predictions : {len(y_val_pred_over):,}"
)

print(
    f"Test predictions       : {len(y_test_pred_over):,}"
)

print("\n" + "=" * 70)
print("OVERSAMPLED MODEL TRAINING COMPLETED")
print("=" * 70)

TF-IDF + LOGISTIC REGRESSION — OVERSAMPLED

Training samples : 540
TF-IDF features  : 4,600

Oversampling target : 30
class_weight       : None

Model training completed.

Validation predictions : 49
Test predictions       : 49

OVERSAMPLED MODEL TRAINING COMPLETED


In [14]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

val_accuracy_over = accuracy_score(y_val_over, y_val_pred_over)
val_macro_f1_over = f1_score(
    y_val_over,
    y_val_pred_over,
    average="macro",
)
val_weighted_f1_over = f1_score(
    y_val_over,
    y_val_pred_over,
    average="weighted",
)

print("=" * 70)
print("TF-IDF + LOGISTIC REGRESSION — OVERSAMPLED VALIDATION")
print("=" * 70)

print(f"Accuracy    : {val_accuracy_over:.4f}")
print(f"Macro-F1    : {val_macro_f1_over:.4f}")
print(f"Weighted-F1 : {val_weighted_f1_over:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_val_over,
        y_val_pred_over,
        zero_division=0,
    )
)

TF-IDF + LOGISTIC REGRESSION — OVERSAMPLED VALIDATION
Accuracy    : 0.6122
Macro-F1    : 0.2944
Weighted-F1 : 0.5499

Classification Report:
               precision    recall  f1-score   support

      Account       0.83      0.56      0.67         9
Communication       0.00      0.00      0.00         2
     Hardware       1.00      0.25      0.40         4
      Network       1.00      0.43      0.60         7
   RemoteWork       0.00      0.00      0.00         1
     Security       0.00      0.00      0.00         3
     Software       0.54      0.95      0.69        22
     Training       0.00      0.00      0.00         1

     accuracy                           0.61        49
    macro avg       0.42      0.27      0.29        49
 weighted avg       0.62      0.61      0.55        49



In [25]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

test_accuracy = accuracy_score(
    y_test,
    y_test_pred,
)

test_macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0,
)

test_weighted_f1 = f1_score(
    y_test,
    y_test_pred,
    average="weighted",
    zero_division=0,
)

print("=" * 70)
print("SELECTED CLASSICAL MODEL — FINAL TEST EVALUATION")
print("=" * 70)

print("Model: TF-IDF + Logistic Regression")
print("Class weighting: balanced")

print(f"\nAccuracy    : {test_accuracy:.4f}")
print(f"Macro-F1    : {test_macro_f1:.4f}")
print(f"Weighted-F1 : {test_weighted_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        zero_division=0,
    )
)

SELECTED CLASSICAL MODEL — FINAL TEST EVALUATION
Model: TF-IDF + Logistic Regression
Class weighting: balanced

Accuracy    : 0.6327
Macro-F1    : 0.5289
Weighted-F1 : 0.6566

Classification Report:
                precision    recall  f1-score   support

       Account       0.40      0.80      0.53         5
 Communication       0.20      0.33      0.25         3
      Hardware       0.50      1.00      0.67         1
Infrastructure       0.00      0.00      0.00         0
       Network       0.83      0.71      0.77         7
    RemoteWork       1.00      1.00      1.00         3
      Security       0.83      1.00      0.91         5
      Software       0.86      0.50      0.63        24
      Training       0.00      0.00      0.00         1

      accuracy                           0.63        49
     macro avg       0.51      0.59      0.53        49
  weighted avg       0.75      0.63      0.66        49



In [26]:
from pathlib import Path
import joblib

MODEL_DIR = Path(
    r"C:\Users\danie\supportmind-ai"
    r"\backend\app\nlp\ticket_category\models"
) / "supportmind_ticket_category"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Save the selected classical classifier.
joblib.dump(
    lr_balanced,
    MODEL_DIR / "classifier.joblib",
)

# Save the TF-IDF vectorizer fitted on the training data.
joblib.dump(
    tfidf,
    MODEL_DIR / "tfidf_vectorizer.joblib",
)

# Save the label encoder.
joblib.dump(
    label_encoder,
    MODEL_DIR / "label_encoder.joblib",
)

print("=" * 70)
print("TICKET CATEGORY MODEL ARTIFACTS SAVED")
print("=" * 70)

print("Model directory:")
print(MODEL_DIR)

print("\nSaved files:")

for file_path in sorted(MODEL_DIR.iterdir()):
    print(
        f"- {file_path.name} "
        f"({file_path.stat().st_size / 1024:.1f} KB)"
    )

TICKET CATEGORY MODEL ARTIFACTS SAVED
Model directory:
C:\Users\danie\supportmind-ai\backend\app\nlp\ticket_category\models\supportmind_ticket_category

Saved files:
- classifier.joblib (331.7 KB)
- label_encoder.joblib (0.6 KB)
- tfidf_vectorizer.joblib (91.8 KB)


In [27]:
import joblib
from pathlib import Path

MODEL_DIR = Path(
    r"C:\Users\danie\supportmind-ai"
    r"\backend\app\nlp\ticket_category\models"
) / "supportmind_ticket_category"

loaded_classifier = joblib.load(
    MODEL_DIR / "classifier.joblib"
)

loaded_tfidf = joblib.load(
    MODEL_DIR / "tfidf_vectorizer.joblib"
)

loaded_label_encoder = joblib.load(
    MODEL_DIR / "label_encoder.joblib"
)

print("=" * 70)
print("SAVED MODEL RELOAD VERIFICATION")
print("=" * 70)

print("Classifier loaded     :", type(loaded_classifier).__name__)
print("TF-IDF loaded         :", type(loaded_tfidf).__name__)
print("Label encoder loaded  :", type(loaded_label_encoder).__name__)

print("\nClasses:")
print(list(loaded_label_encoder.classes_))

print("\nTF-IDF vocabulary size:",
      len(loaded_tfidf.vocabulary_))

SAVED MODEL RELOAD VERIFICATION
Classifier loaded     : LogisticRegression
TF-IDF loaded         : TfidfVectorizer
Label encoder loaded  : LabelEncoder

Classes:
['Account', 'Communication', 'Hardware', 'Infrastructure', 'Licensing', 'Network', 'Performance', 'RemoteWork', 'Security', 'Software', 'Training']

TF-IDF vocabulary size: 3846


In [28]:
# Transform the held-out test text using the reloaded TF-IDF vectorizer.
X_test_reloaded = loaded_tfidf.transform(X_test)

# Predict using the reloaded classifier.
y_test_pred_reloaded = loaded_classifier.predict(
    X_test_reloaded
)

# Verify predictions match the original in-memory model.
predictions_match = np.array_equal(
    y_test_pred,
    y_test_pred_reloaded,
)

print("=" * 70)
print("FINAL MODEL INTEGRITY CHECK")
print("=" * 70)

print("Test samples          :", len(X_test))
print("Original predictions  :", len(y_test_pred))
print("Reloaded predictions  :", len(y_test_pred_reloaded))
print("Predictions identical :", predictions_match)

if not predictions_match:
    differences = np.sum(
        y_test_pred != y_test_pred_reloaded
    )
    print("Prediction differences:", differences)

# Evaluate the reloaded model independently.
reloaded_accuracy = accuracy_score(
    y_test,
    y_test_pred_reloaded,
)

reloaded_macro_f1 = f1_score(
    y_test,
    y_test_pred_reloaded,
    average="macro",
    zero_division=0,
)

reloaded_weighted_f1 = f1_score(
    y_test,
    y_test_pred_reloaded,
    average="weighted",
    zero_division=0,
)

print("\nReloaded model metrics:")
print(f"Accuracy    : {reloaded_accuracy:.4f}")
print(f"Macro-F1    : {reloaded_macro_f1:.4f}")
print(f"Weighted-F1 : {reloaded_weighted_f1:.4f}")

FINAL MODEL INTEGRITY CHECK
Test samples          : 49
Original predictions  : 49
Reloaded predictions  : 49
Predictions identical : True

Reloaded model metrics:
Accuracy    : 0.6327
Macro-F1    : 0.5289
Weighted-F1 : 0.6566


In [29]:
import pandas as pd

# -------------------------------------------------------------------
# Sample customer/support questions covering all 11 categories.
# These are NEW examples and are not taken from the test dataset.
# -------------------------------------------------------------------

sample_questions = [
    {
        "expected_category": "Account",
        "text": (
            "I cannot log in to my account because my password reset "
            "email never arrives."
        ),
    },
    {
        "expected_category": "Communication",
        "text": (
            "My company email is not sending messages to external "
            "customers."
        ),
    },
    {
        "expected_category": "Hardware",
        "text": (
            "My laptop keyboard has stopped working and several keys "
            "are not responding."
        ),
    },
    {
        "expected_category": "Infrastructure",
        "text": (
            "The office server is unavailable and employees cannot "
            "access the shared infrastructure."
        ),
    },
    {
        "expected_category": "Licensing",
        "text": (
            "I need a new software license because my current license "
            "has expired."
        ),
    },
    {
        "expected_category": "Network",
        "text": (
            "I am connected to the office Wi-Fi but I cannot access "
            "the internet."
        ),
    },
    {
        "expected_category": "Performance",
        "text": (
            "The application is extremely slow and takes several "
            "minutes to load each page."
        ),
    },
    {
        "expected_category": "RemoteWork",
        "text": (
            "I am working from home and cannot connect to the company "
            "VPN."
        ),
    },
    {
        "expected_category": "Security",
        "text": (
            "I received a suspicious email asking me to click a link "
            "and provide my company password."
        ),
    },
    {
        "expected_category": "Software",
        "text": (
            "The application crashes whenever I try to open the "
            "reports section."
        ),
    },
    {
        "expected_category": "Training",
        "text": (
            "I need help learning how to use the new employee "
            "management software."
        ),
    },
]

sample_df = pd.DataFrame(sample_questions)

# -------------------------------------------------------------------
# Use ONLY the saved/reloaded artifacts.
# -------------------------------------------------------------------

X_sample = loaded_tfidf.transform(
    sample_df["text"]
)

predicted_labels = loaded_classifier.predict(
    X_sample
)

sample_df["predicted_category"] = predicted_labels

# Prediction confidence.
prediction_probabilities = loaded_classifier.predict_proba(
    X_sample
)

sample_df["confidence"] = (
    prediction_probabilities.max(axis=1)
)

# -------------------------------------------------------------------
# Display results.
# -------------------------------------------------------------------

print("=" * 90)
print("SUPPORTMIND IT TICKET CATEGORY — INFERENCE TEST")
print("=" * 90)

display(
    sample_df[
        [
            "expected_category",
            "predicted_category",
            "confidence",
            "text",
        ]
    ]
)

# -------------------------------------------------------------------
# Summary.
# -------------------------------------------------------------------

correct = (
    sample_df["expected_category"]
    == sample_df["predicted_category"]
).sum()

total = len(sample_df)

print("\nInference summary")
print("-" * 40)
print("Total samples :", total)
print("Correct       :", correct)
print("Accuracy      :", f"{correct / total:.2%}")

SUPPORTMIND IT TICKET CATEGORY — INFERENCE TEST


,expected_category,predicted_category,confidence,text
0,Account,Account,0.295107,I cannot log in to my account because my passw...
1,Communication,Communication,0.151543,My company email is not sending messages to ex...
2,Hardware,Account,0.128805,My laptop keyboard has stopped working and sev...
3,Infrastructure,Network,0.197100,The office server is unavailable and employees...
4,Licensing,Account,0.140948,I need a new software license because my curre...
5,Network,Account,0.181484,I am connected to the office Wi-Fi but I canno...
6,Performance,Performance,0.255253,The application is extremely slow and takes se...
7,RemoteWork,RemoteWork,0.158379,I am working from home and cannot connect to t...
8,Security,Account,0.172172,I received a suspicious email asking me to cli...
9,Software,Performance,0.190941,The application crashes whenever I try to open...



Inference summary
----------------------------------------
Total samples : 11
Correct       : 4
Accuracy      : 36.36%


In [30]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# ============================================================
# WORD-LEVEL TF-IDF
# ============================================================

word_tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
)

X_train_word = word_tfidf.fit_transform(X_train)
X_val_word = word_tfidf.transform(X_val)
X_test_word = word_tfidf.transform(X_test)


# ============================================================
# CHARACTER-LEVEL TF-IDF
# ============================================================

char_tfidf = TfidfVectorizer(
    lowercase=True,
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
)

X_train_char = char_tfidf.fit_transform(X_train)
X_val_char = char_tfidf.transform(X_val)
X_test_char = char_tfidf.transform(X_test)


# ============================================================
# COMBINE WORD + CHARACTER FEATURES
# ============================================================

X_train_combined = hstack(
    [X_train_word, X_train_char]
)

X_val_combined = hstack(
    [X_val_word, X_val_char]
)

X_test_combined = hstack(
    [X_test_word, X_test_char]
)


print("=" * 70)
print("WORD + CHARACTER TF-IDF")
print("=" * 70)

print("Training samples        :", X_train_combined.shape[0])
print("Word features           :", X_train_word.shape[1])
print("Character features      :", X_train_char.shape[1])
print("Combined features       :", X_train_combined.shape[1])


# ============================================================
# LOGISTIC REGRESSION
# ============================================================

lr_word_char = LogisticRegression(
    class_weight="balanced",
    max_iter=3000,
    C=1.0,
    random_state=42,
)

lr_word_char.fit(
    X_train_combined,
    y_train,
)


# ============================================================
# VALIDATION
# ============================================================

y_val_pred_wc = lr_word_char.predict(
    X_val_combined
)

val_accuracy_wc = accuracy_score(
    y_val,
    y_val_pred_wc,
)

val_macro_f1_wc = f1_score(
    y_val,
    y_val_pred_wc,
    average="macro",
    zero_division=0,
)

val_weighted_f1_wc = f1_score(
    y_val,
    y_val_pred_wc,
    average="weighted",
    zero_division=0,
)


print("\nValidation results")
print("-" * 40)
print(f"Accuracy    : {val_accuracy_wc:.4f}")
print(f"Macro-F1    : {val_macro_f1_wc:.4f}")
print(f"Weighted-F1 : {val_weighted_f1_wc:.4f}")

WORD + CHARACTER TF-IDF
Training samples        : 405
Word features           : 11253
Character features      : 11492
Combined features       : 22745

Validation results
----------------------------------------
Accuracy    : 0.7143
Macro-F1    : 0.7017
Weighted-F1 : 0.7271


In [31]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

print("=" * 80)
print("WORD + CHARACTER TF-IDF — VALIDATION CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_val,
        y_val_pred_wc,
        zero_division=0,
    )
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

labels = sorted(y_train.unique())

cm = confusion_matrix(
    y_val,
    y_val_pred_wc,
    labels=labels,
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels,
)

print("\nConfusion Matrix")
print("-" * 80)

display(cm_df)


# ------------------------------------------------------------
# Misclassified examples
# ------------------------------------------------------------

validation_results = pd.DataFrame(
    {
        "text": X_val.reset_index(drop=True),
        "actual": y_val.reset_index(drop=True),
        "predicted": pd.Series(y_val_pred_wc),
    }
)

misclassified = validation_results[
    validation_results["actual"]
    != validation_results["predicted"]
].copy()

print("\nMisclassified validation samples")
print("-" * 80)

print(
    "Total validation samples      :",
    len(validation_results),
)

print(
    "Misclassified samples         :",
    len(misclassified),
)

print(
    "Validation errors             :",
    f"{len(misclassified) / len(validation_results):.2%}",
)

display(misclassified)

WORD + CHARACTER TF-IDF — VALIDATION CLASSIFICATION REPORT
               precision    recall  f1-score   support

      Account       0.80      0.89      0.84         9
Communication       0.25      0.50      0.33         2
     Hardware       1.00      1.00      1.00         4
      Network       0.75      0.86      0.80         7
   RemoteWork       1.00      1.00      1.00         1
     Security       0.50      0.67      0.57         3
     Software       0.86      0.55      0.67        22
     Training       0.25      1.00      0.40         1

     accuracy                           0.71        49
    macro avg       0.68      0.81      0.70        49
 weighted avg       0.79      0.71      0.73        49


Confusion Matrix
--------------------------------------------------------------------------------


,Account,Communication,Hardware,Infrastructure,Licensing,Network,Performance,RemoteWork,Security,Software,Training
Account,8,0,0,0,0,0,0,0,1,0,0
Communication,0,1,0,0,0,0,0,0,0,0,1
Hardware,0,0,4,0,0,0,0,0,0,0,0
Infrastructure,0,0,0,0,0,0,0,0,0,0,0
Licensing,0,0,0,0,0,0,0,0,0,0,0
Network,0,0,0,0,0,6,0,0,0,1,0
Performance,0,0,0,0,0,0,0,0,0,0,0
RemoteWork,0,0,0,0,0,0,0,1,0,0,0
Security,0,0,0,0,0,0,0,0,2,1,0
Software,2,3,0,0,0,2,0,0,1,12,2



Misclassified validation samples
--------------------------------------------------------------------------------
Total validation samples      : 49
Misclassified samples         : 14
Validation errors             : 28.57%


,text,actual,predicted
3,Access O365 Word & Excel Tools Access . : emai...,Software,Account
12,Creating a New Google Calendar for Team Deadli...,Software,Training
15,Fwd: Assistance Required for Troubleshooting G...,Software,Network
17,Email Sync Issues on Mobile Device Hey IT Team...,Software,Communication
18,Intermittent IoT Device Status Reporting Issue...,Network,Software
29,Alert: Intermittent Data Transfer Failures - E...,Security,Software
30,Acme: Firewall Configuration Request: FW-WS-20...,Software,Security
34,Assistance Required for Configuring Multi-Fact...,Account,Security
35,Persistent Sync Issues with Cloud File Service...,Software,Network
36,"Password Reset Assistance Needed Hi IT, I need...",Software,Account


In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

C_VALUES = [0.25, 0.5, 1.0, 2.0, 4.0]

c_results = []

for C in C_VALUES:

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        C=C,
        random_state=42,
    )

    model.fit(
        X_train_combined,
        y_train,
    )

    val_predictions = model.predict(
        X_val_combined
    )

    accuracy = accuracy_score(
        y_val,
        val_predictions,
    )

    macro_f1 = f1_score(
        y_val,
        val_predictions,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        y_val,
        val_predictions,
        average="weighted",
        zero_division=0,
    )

    c_results.append(
        {
            "C": C,
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
        }
    )


c_results_df = pd.DataFrame(c_results)

print("=" * 70)
print("LOGISTIC REGRESSION — C EXPERIMENT")
print("=" * 70)

display(
    c_results_df.sort_values(
        "macro_f1",
        ascending=False,
    ).reset_index(drop=True)
)

LOGISTIC REGRESSION — C EXPERIMENT


,C,accuracy,macro_f1,weighted_f1
0,2.00,0.755102,0.730107,0.768784
1,4.00,0.714286,0.708473,0.727222
2,1.00,0.714286,0.701692,0.727073
3,0.50,0.673469,0.680672,0.683931
4,0.25,0.551020,0.554353,0.551462


In [34]:
from sklearn.metrics import classification_report, confusion_matrix

# Train the selected C=2.0 model
lr_wc_c2 = LogisticRegression(
    class_weight="balanced",
    max_iter=3000,
    C=2.0,
    random_state=42,
)

lr_wc_c2.fit(
    X_train_combined,
    y_train,
)

# Validation predictions
y_val_pred_c2 = lr_wc_c2.predict(
    X_val_combined
)

print("=" * 80)
print("WORD + CHARACTER TF-IDF + LOGISTIC REGRESSION (C=2.0)")
print("=" * 80)

print(
    classification_report(
        y_val,
        y_val_pred_c2,
        zero_division=0,
    )
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

labels = sorted(y_train.unique())

cm_c2 = confusion_matrix(
    y_val,
    y_val_pred_c2,
    labels=labels,
)

cm_c2_df = pd.DataFrame(
    cm_c2,
    index=labels,
    columns=labels,
)

print("\nConfusion Matrix")
print("-" * 80)

display(cm_c2_df)


# ------------------------------------------------------------
# Misclassified examples
# ------------------------------------------------------------

validation_results_c2 = pd.DataFrame(
    {
        "text": X_val.reset_index(drop=True),
        "actual": y_val.reset_index(drop=True),
        "predicted": pd.Series(y_val_pred_c2),
    }
)

misclassified_c2 = validation_results_c2[
    validation_results_c2["actual"]
    != validation_results_c2["predicted"]
].copy()

print("\nMisclassified validation samples")
print("-" * 80)

print("Total validation samples :", len(validation_results_c2))
print("Misclassified samples    :", len(misclassified_c2))
print(
    "Validation errors        :",
    f"{len(misclassified_c2) / len(validation_results_c2):.2%}",
)

display(misclassified_c2)

WORD + CHARACTER TF-IDF + LOGISTIC REGRESSION (C=2.0)
               precision    recall  f1-score   support

      Account       0.80      0.89      0.84         9
Communication       0.25      0.50      0.33         2
     Hardware       1.00      1.00      1.00         4
      Network       0.86      0.86      0.86         7
   RemoteWork       1.00      1.00      1.00         1
     Security       0.50      0.67      0.57         3
     Software       0.88      0.64      0.74        22
     Training       0.33      1.00      0.50         1

     accuracy                           0.76        49
    macro avg       0.70      0.82      0.73        49
 weighted avg       0.81      0.76      0.77        49


Confusion Matrix
--------------------------------------------------------------------------------


,Account,Communication,Hardware,Infrastructure,Licensing,Network,Performance,RemoteWork,Security,Software,Training
Account,8,0,0,0,0,0,0,0,1,0,0
Communication,0,1,0,0,0,0,0,0,0,0,1
Hardware,0,0,4,0,0,0,0,0,0,0,0
Infrastructure,0,0,0,0,0,0,0,0,0,0,0
Licensing,0,0,0,0,0,0,0,0,0,0,0
Network,0,0,0,0,0,6,0,0,0,1,0
Performance,0,0,0,0,0,0,0,0,0,0,0
RemoteWork,0,0,0,0,0,0,0,1,0,0,0
Security,0,0,0,0,0,0,0,0,2,1,0
Software,2,3,0,0,0,1,0,0,1,14,1



Misclassified validation samples
--------------------------------------------------------------------------------
Total validation samples : 49
Misclassified samples    : 12
Validation errors        : 24.49%


,text,actual,predicted
3,Access O365 Word & Excel Tools Access . : emai...,Software,Account
17,Email Sync Issues on Mobile Device Hey IT Team...,Software,Communication
18,Intermittent IoT Device Status Reporting Issue...,Network,Software
29,Alert: Intermittent Data Transfer Failures - E...,Security,Software
30,Acme: Firewall Configuration Request: FW-WS-20...,Software,Security
34,Assistance Required for Configuring Multi-Fact...,Account,Security
35,Persistent Sync Issues with Cloud File Service...,Software,Network
36,"Password Reset Assistance Needed Hi IT, I need...",Software,Account
39,Google Calendar Setup Assistance We need to se...,Software,Training
41,Hi! Can you please provide instructions for su...,Communication,Training


In [35]:
from scipy.sparse import csr_matrix, hstack
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# ============================================================
# FEATURE-WEIGHT EXPERIMENT
# ============================================================

CHAR_WEIGHTS = [0.5, 1.0, 1.5, 2.0]

feature_weight_results = []

for char_weight in CHAR_WEIGHTS:

    # Keep word features unchanged.
    X_train_weighted = hstack(
        [
            X_train_word,
            X_train_char * char_weight,
        ]
    ).tocsr()

    X_val_weighted = hstack(
        [
            X_val_word,
            X_val_char * char_weight,
        ]
    ).tocsr()

    # --------------------------------------------------------
    # Logistic Regression
    # --------------------------------------------------------

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        C=2.0,
        random_state=42,
    )

    model.fit(
        X_train_weighted,
        y_train,
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    predictions = model.predict(
        X_val_weighted
    )

    accuracy = accuracy_score(
        y_val,
        predictions,
    )

    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        y_val,
        predictions,
        average="weighted",
        zero_division=0,
    )

    feature_weight_results.append(
        {
            "char_weight": char_weight,
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
        }
    )


feature_weight_results_df = pd.DataFrame(
    feature_weight_results
)

print("=" * 75)
print("WORD + CHARACTER TF-IDF FEATURE-WEIGHT EXPERIMENT")
print("=" * 75)

display(
    feature_weight_results_df.sort_values(
        "macro_f1",
        ascending=False,
    ).reset_index(drop=True)
)

WORD + CHARACTER TF-IDF FEATURE-WEIGHT EXPERIMENT


,char_weight,accuracy,macro_f1,weighted_f1
0,1.0,0.755102,0.730107,0.768784
1,0.5,0.714286,0.723966,0.717276
2,1.5,0.714286,0.708473,0.727222
3,2.0,0.714286,0.708473,0.727222


In [36]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ============================================================
# FINAL SELECTED MODEL
# ============================================================

final_classifier = LogisticRegression(
    class_weight="balanced",
    max_iter=3000,
    C=2.0,
    random_state=42,
)

final_classifier.fit(
    X_train_combined,
    y_train,
)

# ============================================================
# TEST PREDICTIONS
# ============================================================

y_test_pred_final = final_classifier.predict(
    X_test_combined
)

# ============================================================
# FINAL TEST METRICS
# ============================================================

final_test_accuracy = accuracy_score(
    y_test,
    y_test_pred_final,
)

final_test_macro_f1 = f1_score(
    y_test,
    y_test_pred_final,
    average="macro",
    zero_division=0,
)

final_test_weighted_f1 = f1_score(
    y_test,
    y_test_pred_final,
    average="weighted",
    zero_division=0,
)

print("=" * 80)
print("FINAL IT TICKET CATEGORY CLASSIFIER — TEST EVALUATION")
print("=" * 80)

print("\nModel configuration")
print("-" * 40)
print("Word TF-IDF              : enabled")
print("Character TF-IDF        : enabled")
print("Character weight        : 1.0")
print("Classifier               : LogisticRegression")
print("Class weight             : balanced")
print("C                        : 2.0")

print("\nFinal test metrics")
print("-" * 40)
print(f"Accuracy                 : {final_test_accuracy:.4f}")
print(f"Macro-F1                 : {final_test_macro_f1:.4f}")
print(f"Weighted-F1              : {final_test_weighted_f1:.4f}")

print("\nClassification report")
print("-" * 80)

print(
    classification_report(
        y_test,
        y_test_pred_final,
        zero_division=0,
    )
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

labels = sorted(y_train.unique())

cm_final = confusion_matrix(
    y_test,
    y_test_pred_final,
    labels=labels,
)

cm_final_df = pd.DataFrame(
    cm_final,
    index=labels,
    columns=labels,
)

print("\nConfusion Matrix")
print("-" * 80)

display(cm_final_df)

FINAL IT TICKET CATEGORY CLASSIFIER — TEST EVALUATION

Model configuration
----------------------------------------
Word TF-IDF              : enabled
Character TF-IDF        : enabled
Character weight        : 1.0
Classifier               : LogisticRegression
Class weight             : balanced
C                        : 2.0

Final test metrics
----------------------------------------
Accuracy                 : 0.7755
Macro-F1                 : 0.6902
Weighted-F1              : 0.7756

Classification report
--------------------------------------------------------------------------------
               precision    recall  f1-score   support

      Account       0.50      0.60      0.55         5
Communication       0.33      0.33      0.33         3
     Hardware       1.00      1.00      1.00         1
      Network       0.88      1.00      0.93         7
   RemoteWork       1.00      1.00      1.00         3
     Security       0.83      1.00      0.91         5
     Software      

,Account,Communication,Hardware,Infrastructure,Licensing,Network,Performance,RemoteWork,Security,Software,Training
Account,3,0,0,0,0,0,0,0,0,2,0
Communication,0,1,0,0,0,0,0,0,0,1,1
Hardware,0,0,1,0,0,0,0,0,0,0,0
Infrastructure,0,0,0,0,0,0,0,0,0,0,0
Licensing,0,0,0,0,0,0,0,0,0,0,0
Network,0,0,0,0,0,7,0,0,0,0,0
Performance,0,0,0,0,0,0,0,0,0,0,0
RemoteWork,0,0,0,0,0,0,0,3,0,0,0
Security,0,0,0,0,0,0,0,0,5,0,0
Software,3,2,0,0,0,1,0,0,0,18,0


In [37]:
from sklearn.metrics import classification_report
import pandas as pd

# ============================================================
# FINAL TEST ERROR ANALYSIS
# ============================================================

test_results = pd.DataFrame(
    {
        "text": X_test.reset_index(drop=True),
        "actual": y_test.reset_index(drop=True),
        "predicted": pd.Series(y_test_pred_final),
    }
)

test_results["correct"] = (
    test_results["actual"]
    == test_results["predicted"]
)

print("=" * 80)
print("FINAL TEST CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_test,
        y_test_pred_final,
        zero_division=0,
    )
)


# ============================================================
# MISCLASSIFICATIONS
# ============================================================

test_errors = test_results[
    ~test_results["correct"]
].copy()

print("\n" + "=" * 80)
print("TEST MISCLASSIFICATIONS")
print("=" * 80)

print(
    "Total test samples :",
    len(test_results),
)

print(
    "Correct            :",
    test_results["correct"].sum(),
)

print(
    "Incorrect          :",
    (~test_results["correct"]).sum(),
)

display(test_errors)


# ============================================================
# CONFUSION PAIRS
# ============================================================

confusion_pairs = (
    test_errors
    .groupby(
        ["actual", "predicted"]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("MOST COMMON CONFUSION PAIRS")
print("=" * 80)

display(confusion_pairs)

FINAL TEST CLASSIFICATION REPORT
               precision    recall  f1-score   support

      Account       0.50      0.60      0.55         5
Communication       0.33      0.33      0.33         3
     Hardware       1.00      1.00      1.00         1
      Network       0.88      1.00      0.93         7
   RemoteWork       1.00      1.00      1.00         3
     Security       0.83      1.00      0.91         5
     Software       0.86      0.75      0.80        24
     Training       0.00      0.00      0.00         1

     accuracy                           0.78        49
    macro avg       0.67      0.71      0.69        49
 weighted avg       0.78      0.78      0.78        49


TEST MISCLASSIFICATIONS
Total test samples : 49
Correct            : 38
Incorrect          : 11


,text,actual,predicted,correct
1,Instructions for Email Archiving on Mobile Hi ...,Software,Communication,False
3,Access T-List Board Collaboration 10/15/23 Add...,Account,Software,False
4,cant access gsuite email and calendar hi i nee...,Account,Software,False
10,Access Zoom Webinar Hosting Privileges Hey IT ...,Software,Communication,False
13,"Email Signature Setup Instructions Hey IT, Cou...",Communication,Training,False
15,Assistance Required for Configuring New Hire's...,Software,Account,False
23,Alert: Intermittent Verification Failures in C...,Software,Network,False
35,Phishing Email Reporting Instructions Hello! I...,Training,Security,False
38,Access MS Teams and OneDrive Integration Hi IT...,Software,Account,False
41,Email Distribution List Issue - Intermittent D...,Communication,Software,False



MOST COMMON CONFUSION PAIRS


,actual,predicted,count
0,Software,Account,3
1,Software,Communication,2
2,Account,Software,2
3,Communication,Training,1
4,Communication,Software,1
5,Software,Network,1
6,Training,Security,1


In [38]:
from pathlib import Path
import joblib

# ============================================================
# FINAL MODEL DIRECTORY
# ============================================================

MODEL_DIR = Path(
    r"C:\Users\danie\supportmind-ai"
    r"\backend\app\nlp\ticket_category\models"
) / "supportmind_ticket_category"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# SAVE FINAL ARTIFACTS
# ============================================================

joblib.dump(
    final_classifier,
    MODEL_DIR / "classifier.joblib",
)

joblib.dump(
    word_tfidf,
    MODEL_DIR / "word_tfidf_vectorizer.joblib",
)

joblib.dump(
    char_tfidf,
    MODEL_DIR / "char_tfidf_vectorizer.joblib",
)

joblib.dump(
    label_encoder,
    MODEL_DIR / "label_encoder.joblib",
)

print("=" * 80)
print("FINAL MODEL SAVED")
print("=" * 80)

print("Model directory:")
print(MODEL_DIR)

print("\nSaved files:")

for file_path in sorted(MODEL_DIR.glob("*.joblib")):
    print(
        f"{file_path.name:<35}"
        f"{file_path.stat().st_size / 1024:.1f} KB"
    )

FINAL MODEL SAVED
Model directory:
C:\Users\danie\supportmind-ai\backend\app\nlp\ticket_category\models\supportmind_ticket_category

Saved files:
char_tfidf_vectorizer.joblib       204.4 KB
classifier.joblib                  1955.9 KB
label_encoder.joblib               0.6 KB
tfidf_vectorizer.joblib            91.8 KB
word_tfidf_vectorizer.joblib       282.0 KB


In [39]:
import joblib
from pathlib import Path

MODEL_DIR = Path(
    r"C:\Users\danie\supportmind-ai"
    r"\backend\app\nlp\ticket_category\models"
) / "supportmind_ticket_category"

loaded_final_classifier = joblib.load(
    MODEL_DIR / "classifier.joblib"
)

loaded_word_tfidf = joblib.load(
    MODEL_DIR / "word_tfidf_vectorizer.joblib"
)

loaded_char_tfidf = joblib.load(
    MODEL_DIR / "char_tfidf_vectorizer.joblib"
)

loaded_label_encoder = joblib.load(
    MODEL_DIR / "label_encoder.joblib"
)

print("=" * 80)
print("FINAL MODEL RELOAD VERIFICATION")
print("=" * 80)

print(
    "Classifier loaded      :",
    type(loaded_final_classifier).__name__,
)

print(
    "Word TF-IDF loaded     :",
    type(loaded_word_tfidf).__name__,
)

print(
    "Character TF-IDF loaded:",
    type(loaded_char_tfidf).__name__,
)

print(
    "Label encoder loaded   :",
    type(loaded_label_encoder).__name__,
)

print("\nClasses:")
print(list(loaded_label_encoder.classes_))

print("\nWord vocabulary size    :",
      len(loaded_word_tfidf.vocabulary_))

print("Character vocabulary size:",
      len(loaded_char_tfidf.vocabulary_))

print("\nClassifier C             :",
      loaded_final_classifier.C)

print("Class weight             :",
      loaded_final_classifier.class_weight)

FINAL MODEL RELOAD VERIFICATION
Classifier loaded      : LogisticRegression
Word TF-IDF loaded     : TfidfVectorizer
Character TF-IDF loaded: TfidfVectorizer
Label encoder loaded   : LabelEncoder

Classes:
['Account', 'Communication', 'Hardware', 'Infrastructure', 'Licensing', 'Network', 'Performance', 'RemoteWork', 'Security', 'Software', 'Training']

Word vocabulary size    : 11253
Character vocabulary size: 11492

Classifier C             : 2.0
Class weight             : balanced


In [40]:
from scipy.sparse import hstack
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# ============================================================
# TRANSFORM TEST DATA USING RELOADED VECTORIZERS
# ============================================================

X_test_word_reloaded = loaded_word_tfidf.transform(
    X_test
)

X_test_char_reloaded = loaded_char_tfidf.transform(
    X_test
)

X_test_combined_reloaded = hstack(
    [
        X_test_word_reloaded,
        X_test_char_reloaded,
    ]
).tocsr()

# ============================================================
# PREDICT USING RELOADED CLASSIFIER
# ============================================================

y_test_pred_reloaded_final = (
    loaded_final_classifier.predict(
        X_test_combined_reloaded
    )
)

# ============================================================
# INTEGRITY CHECK
# ============================================================

predictions_match = np.array_equal(
    y_test_pred_final,
    y_test_pred_reloaded_final,
)

accuracy_reloaded = accuracy_score(
    y_test,
    y_test_pred_reloaded_final,
)

macro_f1_reloaded = f1_score(
    y_test,
    y_test_pred_reloaded_final,
    average="macro",
    zero_division=0,
)

weighted_f1_reloaded = f1_score(
    y_test,
    y_test_pred_reloaded_final,
    average="weighted",
    zero_division=0,
)

print("=" * 80)
print("FINAL SAVED MODEL INTEGRITY CHECK")
print("=" * 80)

print("Predictions identical :", predictions_match)

print("\nReloaded model test metrics")
print("-" * 40)

print(
    f"Accuracy    : {accuracy_reloaded:.4f}"
)

print(
    f"Macro-F1    : {macro_f1_reloaded:.4f}"
)

print(
    f"Weighted-F1 : {weighted_f1_reloaded:.4f}"
)

FINAL SAVED MODEL INTEGRITY CHECK
Predictions identical : True

Reloaded model test metrics
----------------------------------------
Accuracy    : 0.7755
Macro-F1    : 0.6902
Weighted-F1 : 0.7756


In [41]:
import pandas as pd
from scipy.sparse import hstack

# ============================================================
# FRESH UNSEEN SUPPORT QUESTIONS
# ============================================================

new_tickets = [
    "My laptop screen is flickering and the display goes black randomly.",

    "I forgot my password and the account recovery link is not working.",

    "The office Wi-Fi keeps disconnecting every few minutes.",

    "I received a suspicious message asking me to verify my company credentials.",

    "The application takes a very long time to open and becomes unresponsive.",

    "I cannot access the company VPN while working from home.",

    "I need assistance getting a license for the analytics software.",

    "My Outlook emails are stuck in the outbox and are not being delivered.",

    "Can someone teach me how to create reports using the new system?",

    "The shared server is unavailable and our team cannot access the files.",

    "I cannot access the project workspace in our collaboration application.",
]

# ============================================================
# WORD FEATURES
# ============================================================

X_new_word = loaded_word_tfidf.transform(
    new_tickets
)

# ============================================================
# CHARACTER FEATURES
# ============================================================

X_new_char = loaded_char_tfidf.transform(
    new_tickets
)

# ============================================================
# COMBINE FEATURES
# ============================================================

X_new_combined = hstack(
    [
        X_new_word,
        X_new_char,
    ]
).tocsr()

# ============================================================
# PREDICTIONS
# ============================================================

new_predictions = loaded_final_classifier.predict(
    X_new_combined
)

# Logistic Regression confidence
new_probabilities = (
    loaded_final_classifier.predict_proba(
        X_new_combined
    )
)

new_confidence = new_probabilities.max(
    axis=1
)

# ============================================================
# RESULTS
# ============================================================

inference_results = pd.DataFrame(
    {
        "text": new_tickets,
        "predicted_category": new_predictions,
        "confidence": new_confidence,
    }
)

print("=" * 100)
print("SUPPORTMIND — FINAL IT CATEGORY INFERENCE")
print("=" * 100)

display(
    inference_results
)

SUPPORTMIND — FINAL IT CATEGORY INFERENCE


,text,predicted_category,confidence
0,My laptop screen is flickering and the display...,Software,0.219197
1,I forgot my password and the account recovery ...,Account,0.581338
2,The office Wi-Fi keeps disconnecting every few...,Hardware,0.307814
3,I received a suspicious message asking me to v...,Software,0.200456
4,The application takes a very long time to open...,Performance,0.470625
5,I cannot access the company VPN while working ...,Account,0.192284
6,I need assistance getting a license for the an...,Software,0.244565
7,My Outlook emails are stuck in the outbox and ...,Account,0.268170
8,Can someone teach me how to create reports usi...,Software,0.263550
9,The shared server is unavailable and our team ...,Account,0.276750
